<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# ToF Basics

For a Duckiedrone to hold its height above the floor, it needs a height estimate. The closest sensor measurement it has is a small laser rangefinder on its underside, pointed at the floor. The rangefinder measures the distance to the surface below it. When the Duckiedrone is level over a bare floor, that distance equals its height, so the flight controller can use it as a height cue.

This rangefinder is a __time-of-flight (ToF) sensor__: it measures the distance to a surface by emitting light and detecting the light that comes back. The Duckiedrone carries five of them: one along $+z$ (down), one along $-z$ (up), and three horizontally along $+x$ (front), $-y$ (left), and $+y$ (right). These are the Duckiedrone body axes: $+x$ points forward, $+y$ right, and $+z$ down. All five use the same chip, the [VL53L1X](https://www.st.com/en/imaging-and-photonics-solutions/vl53l1x.html), made by STMicroelectronics (ST).

[Figure 1](#figure-1) shows where the five sensors sit and the axis each one points along.

<figure id="figure-1" class="lx-figure" style="width:640px;">
  <img src="../assets/_images/tof_sensor_layout.svg" style="width:100%;" />
  <figcaption>
    Figure 1: The five ToF sensors and the Duckiedrone body axes.
  </figcaption>
</figure>

A ToF sensor does not always return a distance. It reports one only when enough of its light comes back, and the distance at which that stops happening depends on the surface it points at and on how bright the room is.

By the end of the notebook you should be able to answer:

- What does a ToF sensor measure, and how does the VL53L1X turn detected light into a distance?

- Why does the distance reported by a stationary sensor change from one reading to the next?

- Why is the maximum range not a fixed number, and what sets it?

- Why do the errors of a rangefinder not accumulate the way the errors of an integrated
  acceleration do?

## Notation

[Table 1](#table-1) lists the mathematical conventions, [Table 2](#table-2) the spaces and structures, and [Table 3](#table-3) the semantic symbols used throughout this learning experience (LX).

<a id="table-1"></a>

Table 1: Mathematical conventions used in equations throughout this LX.

| Notation | Meaning |
| --- | --- |
| $\tilde{(\cdot)}$ | Reported measurement of the underlying physical quantity |
| $\Delta(\cdot)$ | Difference between two values of the enclosed quantity |
| $(\cdot)^2$, $\sqrt{\cdot}$ | Square and non-negative square root of the enclosed quantity |
| $a \times b$ | Product of $a$ and $b$ |
| $\frac{a}{b}$ | Quotient of $a$ and nonzero $b$ |
| $\{\cdots\}$ | Set containing the listed elements |
| $=$, $\approx$, $\in$ | Equality, approximate equality, and set membership |
| $<$, $\leq$, $>$, $\geq$ | Strict and non-strict order relations |

[Table 2](#table-2) lists the spaces and structures used throughout this LX.

<a id="table-2"></a>

Table 2: Spaces and structures used throughout this LX.

| Space or structure | Meaning |
| --- | --- |
| $\mathbb{Z}_{\geq 0}$ | Non-negative integers, $\{0, 1, 2, \ldots\}$ |
| $\mathbb{R}_{\geq 0}$ | Non-negative real numbers |
| $\mathbb{R}_{>0}$ | Positive real numbers |

[Table 3](#table-3) defines the semantic symbols used in the equations.

<a id="table-3"></a>

Table 3: Semantic symbols used in equations throughout this LX.

| Symbol | Meaning | Type | Units |
| --- | --- | --- | --- |
| $d$ | Physical distance from the sensor to the reflecting surface, along the sensor's axis | $d \in \mathbb{R}_{\geq 0}$ | $\mathrm{m}$ |
| $\tilde{d}$ | Reported range when a target is detected; a noisy measurement of $d$ | $\tilde{d} \in \mathbb{R}_{\geq 0}$ | $\mathrm{m}$ |
| $c$ | Speed of light in air, approximately $3 \times 10^8\,\mathrm{m/s}$ | $c \in \mathbb{R}_{>0}$ | $\mathrm{m/s}$ |
| $\Delta t$ | Round-trip light-travel time | $\Delta t \in \mathbb{R}_{>0}$ | $\mathrm{s}$ |
| $N$ | Photon count in the tallest histogram bin | $N \in \mathbb{Z}_{\geq 0}$ | $\mathrm{photons}$ |
| $B$ | Estimated background count per bin, the median of all histogram bins | $B \in \mathbb{R}_{\geq 0}$ | $\mathrm{photons/bin}$ |
| $\mathrm{SNR}$ | Signal-to-noise ratio of the tallest histogram bin | $\mathrm{SNR} \in \mathbb{R}_{\geq 0}$ | - |

## What a time-of-flight sensor measures

[Figure 2](#figure-2) illustrates the round trip of light used to measure distance.

<figure id="figure-2" class="lx-figure" style="width:420px;">
  <img src="../assets/_images/Time_of_flight.svg" style="width:100%;" />
  <figcaption>
    Figure 2: The principle of time-of-flight ranging. Light leaves the sensor, reflects from the
    target, and returns, so its travel time covers the distance twice. See the <a
    href="https://en.wikipedia.org/wiki/Time-of-flight_camera#/media/File:20200501_Time_of_flight.svg">image source</a>.
  </figcaption>
</figure>

The source graphic labels the full emission-to-reception duration $t$. This notebook writes the same interval as $\Delta t$ to distinguish it from an absolute timestamp.

Let $c$ be the speed of light in air and $d$ the physical distance from the sensor to the reflecting surface along its axis. As [Figure 2](#figure-2) shows, light travels from the sensor to that surface and back, so the round-trip time $\Delta t$ covers $d$ twice:

<span id="equation-1"></span>

$$
d = \frac{c \, \Delta t}{2}.
\tag{1}
$$

A sensor report is a measurement of that distance, written $\tilde{d}$ below; it can differ from $d$ because the detected photons are random.

The VL53L1X does not look along a thin line. It collects light from a cone about $27^\circ$ wide, so $d$ is the distance to the surface inside that cone that returns the light it detects. Its [datasheet](https://www.st.com/resource/en/datasheet/vl53l1x.pdf) gives the full specification.

## Gathering ToF measurements from a Duckiedrone

Before looking at how that round-trip time is obtained, it is worth looking at what the sensor reports.

Each reading from the downward sensor arrives as one ROS 2 message. The command below prints one message, where `DUCKIEDRONE_NAME` is the name of your Duckiedrone:

```bash
ros2 topic echo --once /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

This section only shows what the data looks like. Running the command needs a shell on the Duckiedrone, which [Notebook 3](./3-subscribing-to-tof.ipynb#getting-to-a-terminal-that-can-see-the-duckiedrone) sets up; follow it now to explore the data ahead. A virtual Duckiedrone publishes the same message, with distances computed by the simulator instead of measured. The simulated distance carries no noise: a virtual Duckiedrone at rest reports the same value in every message. The [Duckiematrix](https://docs.duckietown.com/ente/duckietown-manual/50-duckiematrix/getting-started/duckiematrix-first-steps.html) has to be running, with the Duckiedrone attached to an entity, or the topic stays silent.

The message below was captured on a physical Duckiedrone resting on the floor:

```shell
header:
  stamp:
    sec: 1787340653
    nanosec: 105715314
  frame_id: pdrone24/tof/bottom
radiation_type: 1
field_of_view: 0.471238911151886
min_range: 0.03999999910593033
max_range: 3.5999999046325684
range: 0.017999999225139618
variance: 0.0
```

This message is of the type `sensor_msgs/Range`. Its `range` field carries a reported range: a measurement $\tilde{d}$ of the physical distance $d$ in $\mathrm{m}$ when the driver has one. When no usable range is available, the bridge uses `999.0` instead, as [Notebook 2](./2-duckiedrone-tof-architecture.ipynb#the-ros-2-bridge) explains. The remaining fields describe the sensor rather than this particular measurement: `field_of_view` is the $27^\circ$ cone, in radians, for example.

The sensor sits about $2\,\mathrm{cm}$ above the floor, so a reading of $0.018\,\mathrm{m}$ is what one would expect. It is below the `min_range` of $0.04\,\mathrm{m}$ in the same message, the closest distance ST specifies for the VL53L1X, so the sensor still reports distances below its specified minimum. On a physical Duckiedrone, running the same command again usually gives a slightly different value, even though nothing has moved. This variation comes from how the VL53L1X measures distance, which the rest of this notebook explains.

## How the distance is measured

Light travels about $30\,\mathrm{cm}$ in $1\,\mathrm{ns}$. Because the pulse travels the distance twice, [Equation (1)](#equation-1) means that a $1\,\mathrm{cm}$ change in distance changes the round-trip time by about $67\,\mathrm{ps}$. The VL53L1X reports distances in whole $1\,\mathrm{mm}$ increments, ten times finer still. Resolving $1\,\mathrm{mm}$ by timing one reflected pulse would mean measuring its arrival to within about $7\,\mathrm{ps}$.

The VL53L1X does not attempt that. Instead of timing _one_ pulse precisely, it times _a great many_ photons coarsely and combines the results:

1. __It detects single photons.__ The receiver is an array of __single-photon avalanche diodes
   (SPADs)__. One arriving photon triggers an avalanche of current, a clean digital pulse. A SPAD
   does not report how bright the light is, only that a photon arrived, and when.

2. __It builds a histogram.__ The emitter fires a short infrared pulse many times per
   measurement, and every detected photon is counted into a bin according to its arrival time. A
   single photon could come from the target or from sunlight. After many pulses, a bump rises out
   of a flat background of ambient light.

3. __It finds the peak.__ That bump is the target. The chip locates it, converts its position to
   a distance in $\mathrm{mm}$, and reports it.

> The sensor does not time one reflected pulse with a stopwatch: it builds a distribution of
> photon arrival times and reports where its peak is.

## Activity: a model of one measurement

The two cells below model a single measurement: fire many pulses, count the returning photons into bins, and find the peak. Each bin is $67\,\mathrm{ps}$ wide, the round-trip time for $1\,\mathrm{cm}$ of distance by [Equation (1)](#equation-1), so the bins are labeled in $\mathrm{m}$.

The model is deliberately simplified. It assumes one target filling the whole cone, a returning pulse with a smooth bell shape, background light spread evenly over every bin, and a peak taken as the tallest bin. Its constants were chosen to give numbers of the right size, not fitted to a VL53L1X, so its distances are illustrations rather than predictions.

What it keeps are the three ingredients the rest of this notebook needs: the signal from the target weakens with distance, as $\frac{1}{d^2}$; the background does not; and photons arrive at random, so every count comes with noise.

[Table 4](#table-4) lists the inputs to the one-measurement model.

<a id="table-4"></a>

Table 4: Inputs to the one-measurement model.

| Input | Meaning |
| --- | --- |
| `distance_m` | Physical distance from the sensor to the target, in $\mathrm{m}$ |
| `reflectivity` | Fraction of the arriving light the target sends back, from $0$ to $1$ |
| `ambient_suns` | Background light, as a multiple of direct sunlight: $1$ is full sun, and indoor lighting is a small fraction of $1$ |
| `mode`, `timing_budget_ms` | Two chip settings, explained in the [next section](#the-chips-settings); this activity leaves them at `LONG_RANGE` and $33\,\mathrm{ms}$ |
| `seed` | Selects one set of random photon arrivals, so the same inputs always give the same result |


In [ ]:
# The sensor model. This cell defines functions and produces no plot.
import matplotlib.pyplot as plt
import numpy as np

# Histogram bins. One bin is 67 ps of round-trip time, which is 1 cm of distance.
N_BINS = 400
BIN_WIDTH_M = 0.01
BIN_DISTANCE = (
    np.arange(N_BINS) + 0.5
) * BIN_WIDTH_M  # Center of each bin, in m.
PULSE_SPREAD_M = (
    0.045  # Width of the returning pulse, as a standard deviation, in m.
)

# Distance mode: how long after each pulse the chip still accepts returning photons, written as
# the distance that time corresponds to.
MODE_WINDOW_M = {"SHORT_RANGE": 1.36, "LONG_RANGE": 3.60}

# Timing budgets the chip accepts, in ms. 15 ms exists in SHORT_RANGE only.
LEGAL_BUDGETS_MS = {
    "SHORT_RANGE": (15, 20, 33, 50, 100, 200, 500),
    "LONG_RANGE": (20, 33, 50, 100, 200, 500),
}

# The setting the photon gains below are given for: LONG_RANGE with a 33 ms timing budget.
REF_WINDOW_M = 3.60
REF_BUDGET_MS = 33.0

SIGNAL_GAIN = 13000.0  # Signal photons from a fully reflective target at 1 m.
AMBIENT_GAIN = 15.0  # Background photons per bin in direct sunlight.
DETECTION_SNR = 6.0  # Minimum SNR for a peak to count as a target. SNR is defined below the plot.


def photon_histogram(  # noqa: PLR0913
    distance_m,
    reflectivity=0.5,
    ambient_suns=0.1,
    mode="LONG_RANGE",
    timing_budget_ms=33.0,
    seed=0,
):
    """Simulate the photons collected during one measurement.

    Returns the distance of each bin, in m, and the number of photons counted in it.
    """
    window_m = MODE_WINDOW_M[mode]
    bins = BIN_DISTANCE[
        : round(window_m / BIN_WIDTH_M)
    ]  # The chip histograms no further.

    # A longer budget collects proportionally more photons of every kind.
    budget_factor = timing_budget_ms / REF_BUDGET_MS

    # A shorter window means less time waiting after each pulse, so more pulses fit in the budget.
    pulse_factor = REF_WINDOW_M / window_m

    # Ambient light arrives at a constant rate, so it fills every bin equally and carries no
    # information about distance. More pulses collect more of it too.
    ambient = AMBIENT_GAIN * ambient_suns * budget_factor * pulse_factor
    expected = np.full_like(bins, ambient)

    # The window is a hard edge: a target beyond it contributes no signal photons.
    if distance_m <= window_m:
        # The number of photons that make it back falls off as 1/d^2 and scales with how much of
        # the light the target reflects. The distance is floored at 1 mm to avoid dividing by zero.
        distance_sq = max(distance_m, 1e-3) ** 2
        n_signal = (
            SIGNAL_GAIN
            * reflectivity
            * budget_factor
            * pulse_factor
            / distance_sq
        )

        # Spread those photons over the bins as a bell curve centered on the true distance.
        # Dividing by the curve's area, measured in bins, makes the bump hold n_signal photons.
        bump = np.exp(-0.5 * ((bins - distance_m) / PULSE_SPREAD_M) ** 2)
        expected += (
            n_signal
            * bump
            / (np.sqrt(2 * np.pi) * PULSE_SPREAD_M / BIN_WIDTH_M)
        )

    # Photons arrive at random, which is where the noise comes from.
    rng = np.random.default_rng(seed)
    return bins, rng.poisson(expected)


def find_peak(bins, counts):
    """Find the tallest bin and decide whether it is a target.

    Returns the peak's distance in m (None when its SNR is below DETECTION_SNR), the background
    level, and the peak's SNR.
    """
    # Most bins hold only background light, so their median is the background level.
    background = np.median(counts)
    peak = int(np.argmax(counts))

    # The +1 keeps the SNR defined when every bin is empty, with no ambient light and no target.
    snr = (counts[peak] - background) / np.sqrt(
        counts[peak] + background + 1.0
    )
    if snr < DETECTION_SNR:
        return None, background, snr
    return bins[peak], background, snr

In [ ]:
X_MAX_M = 4.0  # Right edge of the plot, in m.


def show_measurement(  # noqa: PLR0913
    distance_m=1.0,
    reflectivity=0.5,
    ambient_suns=0.1,
    mode="LONG_RANGE",
    timing_budget_ms=33.0,
    seed=0,
):
    """Simulate one measurement and plot its histogram, the true distance, and the estimate."""
    if timing_budget_ms not in LEGAL_BUDGETS_MS[mode]:
        print(  # noqa: T201
            f"The VL53L1X does not accept this: {timing_budget_ms:.0f} ms is not one of "
            f"{LEGAL_BUDGETS_MS[mode]} in {mode}."
        )
        return

    window_m = MODE_WINDOW_M[mode]
    bins, counts = photon_histogram(
        distance_m, reflectivity, ambient_suns, mode, timing_budget_ms, seed
    )
    estimate, background, snr = find_peak(bins, counts)

    _, ax = plt.subplots(figsize=(9, 4))
    ax.step(
        bins,
        counts,
        where="mid",
        color="tab:blue",
        lw=1,
        label="photons detected",
    )
    ax.axhline(
        background, color="tab:gray", ls=":", label="ambient background"
    )
    ax.axvline(distance_m, color="k", ls="--", lw=1, label="true distance")

    # Shade the distances beyond the end of the window, where no photon is counted.
    ax.axvspan(window_m, X_MAX_M, color="0.93", zorder=0)
    ax.axvline(
        window_m,
        color="0.55",
        ls="-.",
        lw=1.2,
        label=f"end of window, {window_m:.2f} m",
    )

    if estimate is None:
        ax.set_title(
            f"no target detected    peak SNR {snr:.1f}, needs {DETECTION_SNR:.0f}"
        )
    else:
        ax.axvline(estimate, color="tab:red", lw=1.5, label="sensor estimate")
        ax.set_title(
            f"reports {estimate:.2f} m    "
            f"error {(estimate - distance_m) * 100:+.1f} cm    "
            f"peak SNR {snr:.0f}"
        )

    ax.set_xlabel("distance [m]   (bin arrival time, converted)")
    ax.set_ylabel("photons in bin")
    ax.set_xlim(0, X_MAX_M)
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)
    plt.show()


show_measurement()

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

The plot shows everything the sensor has to work with, and what the model makes of it:

- The __blue line__ is the number of photons counted in each bin.

- The __flat grey level__ is background light. It fills every bin by the same amount and says
  nothing about distance; it is the noise the signal has to be seen over. The dotted grey line
  marks its level, taken as the median count over all bins.

- The __spike__ is the pulse returning from the target. Its position on the $x$-axis is the
  distance reading.

- The __black dashed line__ is the physical distance to the target, and the __red line__ is the
  distance the model reports, the position of the tallest bin. The gap between the two is the
  error.

- The __shaded region on the right__ lies beyond the __window__, the fixed time the chip waits
  for returning light after each pulse before it fires again. Photons arriving later are not
  counted, so a target out there cannot be detected however strong its return. The dash-dotted
  line marks where the window ends.

- The __title__ gives the reported distance, its error, and the peak's $\mathrm{SNR}$, defined
  below. When the peak is too weak, the title says no target was detected and no red line is drawn.

Whether the spike counts as a target is a question of how far it stands above the noise. Photon counts are random, and for counts like these the typical size of their fluctuation is the square root of the count. The __signal-to-noise ratio__ $\mathrm{SNR}$ compares the peak above background with those fluctuations. With $N$ the photons in the tallest bin and $B$ the background level, the model computes:

<span id="equation-2"></span>

$$
\mathrm{SNR} = \frac{N - B}{\sqrt{N + B}}.
\tag{2}
$$

It reports a distance only when the $\mathrm{SNR}$ reaches $6$. The code adds $1$ under the square root so that the ratio stays defined when every bin is empty. The threshold of $6$ is a choice made for this model, not a value taken from the VL53L1X.

The model takes the tallest bin as the answer, so it cannot be more precise than one $1\,\mathrm{cm}$ bin. The VL53L1X reports distances in whole $1\,\mathrm{mm}$ increments, so its own processing extracts more from the histogram than the tallest bin; the model does not reproduce how.

## The chip's settings

Two settings of the VL53L1X change what a single measurement looks like. Both can be changed in the widget below.

### Distance mode: how long the chip listens

After each pulse the chip accepts returning photons for a fixed time, the window, then fires again. The distance mode sets the length of that window: `SHORT_RANGE` or `LONG_RANGE`. The end of the window is the dash-dotted line on the plot.

A shorter window has two effects:

- __Less reach.__ Light from a target beyond the end of the window returns too late to be
  counted. In the model, short mode detects nothing beyond $1.36\,\mathrm{m}$.

- __More pulses.__ The chip waits less after each pulse, so more pulses fit into one measurement.
  In the model that is $\frac{3.60\,\mathrm{m}}{1.36\,\mathrm{m}} \approx 2.6$ times as many. More
  pulses collect more signal and more background, and the SNR rises by
  $\sqrt{2.6} \approx 1.6$. This helps most in bright light, where the SNR is lowest to begin with.

### Timing budget: how long one measurement collects

The timing budget is how long the chip collects photons into the histogram before it reports. The VL53L1X accepts timing budgets of $\{15, 20, 33, 50, 100, 200, 500\}\,\mathrm{ms}$, with $15\,\mathrm{ms}$ in short mode only, and the Duckiedrone's downward sensor uses $33\,\mathrm{ms}$.

A longer budget collects more photons, signal and background alike. The spike stands further above the noise, so the reading is steadier and the sensor reaches further. The reach grows much more slowly than the budget, and how much it grows depends on the light. In the model, going from $33\,\mathrm{ms}$ to $200\,\mathrm{ms}$, about six times the budget, extends the maximum range:

- about $1.6$ times in bright light, where the background adds noise that also grows with the
  budget;

- about $2.5$ times in the dark, where the only noise comes from the signal itself, until the end
  of the window caps it.

The chip has one more timing setting, the inter-measurement period: the time from the start of one measurement to the start of the next. It cannot be shorter than the timing budget. It changes how often a measurement is made, not what a measurement looks like, so it has no slider here. [Notebook 2](./2-duckiedrone-tof-architecture.ipynb) shows where all three settings are configured on the Duckiedrone.

### Predict, then check

The widget below exposes every input of the model as a control. `distance mode` and `budget [ms]` are the two chip settings. `distance [m]`, `reflectivity`, and `ambient [suns]` describe the situation the Duckiedrone is in rather than anything it can configure. `seed` selects a different set of random photon arrivals.

ST specifies the range of the VL53L1X against two test targets: a white one that reflects $88\%$ of the sensor's infrared light, and a grey one that reflects $17\%$.

The widget starts with a target $2.5\,\mathrm{m}$ away, indoors, in `LONG_RANGE` with a $33\,\mathrm{ms}$ budget. Watch the histogram, not only the number in the title.


In [ ]:
from ipywidgets import Dropdown, FloatSlider, IntSlider, interact

LABEL_STYLE = {"description_width": "100px"}

interact(
    show_measurement,
    distance_m=FloatSlider(
        value=2.5,
        min=0.05,
        max=4.0,
        step=0.05,
        description="distance [m]",
        continuous_update=False,
        style=LABEL_STYLE,
    ),
    reflectivity=FloatSlider(
        value=0.5,
        min=0.01,
        max=1.0,
        step=0.01,
        description="reflectivity",
        continuous_update=False,
        style=LABEL_STYLE,
    ),
    ambient_suns=FloatSlider(
        value=0.1,
        min=0.0,
        max=10.0,
        step=0.1,
        description="ambient [suns]",
        continuous_update=False,
        style=LABEL_STYLE,
    ),
    mode=Dropdown(
        options=["LONG_RANGE", "SHORT_RANGE"],
        value="LONG_RANGE",
        description="distance mode",
        style=LABEL_STYLE,
    ),
    timing_budget_ms=Dropdown(
        options=[15, 20, 33, 50, 100, 200, 500],
        value=33,
        description="budget [ms]",
        style=LABEL_STYLE,
    ),
    seed=IntSlider(
        value=0,
        min=0,
        max=10,
        step=1,
        description="seed",
        continuous_update=False,
        style=LABEL_STYLE,
    ),
);

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

Each step below changes one control. Predict what the histogram will do before moving it.

1. __Distance.__ Move `distance [m]` down to $1.0\,\mathrm{m}$. The spike moves left and grows about
   $5$ times taller, because the returning light follows $\frac{1}{d^2}$. The background does not
   change. Now move it out to $3.6\,\mathrm{m}$: the spike is small but still detected. At
   $3.7\,\mathrm{m}$ it falls into the shaded region, beyond the end of the window, and disappears
   entirely. Return to $2.5\,\mathrm{m}$.

2. __Reflectivity.__ Lower `reflectivity` from $0.5$ to $0.2$, close to ST's grey target. The spike
   shrinks to less than $\frac{1}{2}$ of its height, and the title reports no target. Now move `seed`:
   some seeds find the target and others do not. Near the limit of detection, whether a reading
   arrives at all comes down to chance. Return `reflectivity` to $0.5$ and `seed` to $0$.

3. __Ambient light.__ Raise `ambient [suns]` from $0.1$ to $10$, direct sunlight. The flat background
   rises from about $2\,\mathrm{photons/bin}$ to about $150\,\mathrm{photons/bin}$. The spike still
   sits the same height above it, but the background brings its own noise, and the title reports no
   target.

4. __Timing budget.__ Leave the sunlight on and raise `budget [ms]` from $33\,\mathrm{ms}$ to
   $200\,\mathrm{ms}$. Everything grows about $6$ times, spike and background alike, and the target
   is detected again. A longer budget buys back what the sunlight took away. Return `budget [ms]`
   to $33\,\mathrm{ms}$ and `ambient [suns]` to $0.1$.

5. __Distance mode.__ Switch `distance mode` to `SHORT_RANGE`. The shaded region moves in to
   $1.36\,\mathrm{m}$ and covers the target, so nothing is detected. Now move `distance [m]` to
   $1.0\,\mathrm{m}$: the target is back, with a spike almost $3$ times taller than `LONG_RANGE`
   gives at the same distance. Switch back to `LONG_RANGE` to compare.

6. __A budget the chip does not accept.__ In `SHORT_RANGE`, pick a $15\,\mathrm{ms}$ budget: the
   chip accepts it. Now switch to `LONG_RANGE`. The widget refuses, because $15\,\mathrm{ms}$ exists
   in short mode only.

## The same measurement on a physical Duckiedrone

The model illustrates the mechanism, but its constants were chosen rather than measured. The recording below replaces the model with a sensor.

A physical Duckiedrone was placed on the floor of a room, powered up with its motors off, and left static for $3\,\mathrm{min}$ while its downward ToF sensor was logged. The log ships with this LX as a comma-separated values (CSV) file, [assets/data/tof_still_3min.csv](../assets/data/tof_still_3min.csv), with columns $t$, in $\mathrm{s}$ from the start, and `range`, in $\mathrm{m}$. These are recorded measurements, not simulated ones.

The sensor sat about $2\,\mathrm{cm}$ above the floor, which is below the $0.04\,\mathrm{m}$ minimum that ST specifies for the VL53L1X. The readings are still real measurements, but their spread is not guaranteed to match the spread at larger distances.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt  # noqa: F811
import numpy as np

# The notebook may be run from the repository root or from the notebooks folder.
data_path = Path("assets/data/tof_still_3min.csv")
if not data_path.exists():
    data_path = Path("../assets/data/tof_still_3min.csv")

# Downward ToF of a static Duckiedrone, logged for three minutes.
# Columns: t [s since start], range [m].
recording = np.genfromtxt(data_path, delimiter=",", names=True)
t_s = recording["t"]
range_mm = recording["range"] * 1000.0  # Meters to millimeters.
duration_s = t_s[-1] - t_s[0]
rate_hz = (len(t_s) - 1) / duration_s

fig, (ax_t, ax_h) = plt.subplots(1, 2, figsize=(12, 4), width_ratios=[2, 1])

# Left: one marker per reading, not a line, since the recording says nothing about what happened
# between two readings.
ax_t.plot(
    t_s, range_mm, linestyle="none", marker=".", markersize=2, color="tab:blue"
)
ax_t.axhline(
    range_mm.mean(),
    color="k",
    ls="--",
    lw=1,
    label=f"mean {range_mm.mean():.1f} mm",
)
ax_t.set_xlabel("time [s]")
ax_t.set_ylabel("reported range [mm]")
ax_t.set_title(
    f"{len(range_mm)} readings over {duration_s:.0f} s at {rate_hz:.1f} Hz"
)
ax_t.legend(loc="upper right")
ax_t.grid(alpha=0.3)

# Right: how many readings reported each value. The chip reports whole millimeters, so each bar
# is 1 mm wide and centered on one value.
edges = np.arange(range_mm.min() - 0.5, range_mm.max() + 1.5, 1.0)
ax_h.hist(range_mm, bins=edges, color="tab:blue", edgecolor="white")
ax_h.set_xlabel("reported range [mm]")
ax_h.set_ylabel("number of readings")
ax_h.set_title(f"standard deviation {range_mm.std():.2f} mm")
ax_h.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(  # noqa: T201
    "distinct values reported [mm]:",
    sorted({int(v) for v in np.round(range_mm)}),
)

# Mean of the first minute against mean of the last minute.
first_minute = range_mm[t_s < t_s[0] + 60.0]
last_minute = range_mm[t_s > t_s[-1] - 60.0]
print(  # noqa: T201
    f"mean of the last minute minus mean of the first minute: "
    f"{last_minute.mean() - first_minute.mean():+.2f} mm"
)

# Correlation between each reading and the next. Near 0, one reading says nothing about the next.
deviation = range_mm - range_mm.mean()
consecutive_corr = np.mean(deviation[:-1] * deviation[1:]) / np.var(range_mm)
print(f"correlation between consecutive readings: {consecutive_corr:.2f}")  # noqa: T201

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

### What the recording shows

The left plot shows every reading against time, with a dashed line at their mean. The right plot is a histogram of the readings themselves, not of photons: each bar counts the readings at one $1\,\mathrm{mm}$ value.

__The reading varies although nothing moved.__ Over $3\,\mathrm{min}$ the reported distance spans $11\,\mathrm{mm}$ to $25\,\mathrm{mm}$, with a standard deviation of about $2\,\mathrm{mm}$. This is the `seed` slider on a physical sensor: each measurement collects a fresh set of random photons, so its peak lands in a slightly different place. The values come in whole $1\,\mathrm{mm}$ steps, the finest step the chip reports.

__The average does not move.__ The means of the first and last minute differ by about $0.25\,\mathrm{mm}$, a small fraction of the spread between single readings.

__One reading says nothing about the next.__ The correlation between consecutive readings is $0.02$, close to zero: a reading above the mean is no more likely than any other to be followed by another reading above the mean.

The reason is what the sensor measures. A __relative__ measurement describes a change from a starting point, so any estimate built from it depends on that starting point and on every change added since. An __absolute__ measurement is taken against a reference outside the sensor, so it does not depend on what came before. A position obtained by integrating accelerometer readings is the first kind: each error is added into a running total, and in the [IMU LX](https://github.com/duckietown/lx-dd-sensors-imu) an altitude computed this way from a physical Duckiedrone went through a $2.5\,\mathrm{m}$ ceiling within $20\,\mathrm{s}$. The ToF sensor is the second kind. Each reading measures the distance to the surface again, from new photons, which is why consecutive readings are uncorrelated, and so its errors do not accumulate.

## What "maximum range" means

The model already answers this, and the answer is not a single number.

A peak is detected when it clears the noise. As the target moves away, fewer photons return, the peak shrinks, and at some distance its SNR falls below the threshold. That distance is where detection stops. The widget showed what moves it: at $2.5\,\mathrm{m}$, a less reflective target or sunlight lost the target, and a longer timing budget brought it back. It is a signal-to-noise limit, not a fixed property of the sensor.

The one hard edge is the distance mode's window, because photons arriving after it closes are not counted however many there are. The window distances in the model, $1.36\,\mathrm{m}$ and $3.60\,\mathrm{m}$, are the figures the Adafruit VL53L1X library gives for the two modes. ST's datasheet quotes up to $1.3\,\mathrm{m}$ for short mode and up to $4\,\mathrm{m}$ for long mode, under favorable conditions.

When the VL53L1X finds no peak that clears the noise, it reports that no target was found rather than a large distance.

## What a ToF sensor does not measure

A ToF sensor on its own does not measure:

- __Altitude__, meaning height above a fixed reference such as the take-off point. It measures
  the distance to whatever surface is below, so the reading drops when the Duckiedrone flies
  over a table, even though its altitude has not changed.

- __Which surface returned the light.__ Anything inside the $27^\circ$ cone can contribute, and
  the sensor reports a single distance.

- __Vertical height when the Duckiedrone tilts.__ The sensor measures along its own axis. When the
  Duckiedrone tilts, the downward sensor tilts with it, and the reading is the longer, slanted
  distance to the floor.

- __Anything it cannot detect.__ A target beyond the window, or too dark for its peak to clear
  the noise, produces no distance at all.

## Further reading

The [VL53L1X datasheet](https://www.st.com/resource/en/datasheet/vl53l1x.pdf) specifies the timing budget, distance modes, and range conditions behind the experiments in this notebook.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()